In [4]:
!pip install -q sentence-transformers

In [5]:
belgeler = [
    "Python, 1991 yılında Guido van Rossum tarafından geliştirilen, okunabilir bir programlama dilidir.",
    "PyTorch, Facebook (Meta) AI Research ekibi tarafından geliştirilen açık kaynaklı bir derin öğrenme kütüphanesidir.",
    "TensorFlow, Google Brain ekibi tarafından geliştirilen bir makine öğrenmesi kütüphanesidir.",
    "Hugging Face, yapay zeka modellerinin paylaşıldığı popüler bir platformdur.",
]

In [3]:
from sentence_transformers import SentenceTransformer, util

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

belge_vektorleri = embed_model.encode(belgeler, convert_to_tensor=True)

print("Belge sayısı:", len(belgeler))
print("Her vektörün boyutu:", belge_vektorleri.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Belge sayısı: 4
Her vektörün boyutu: torch.Size([4, 384])


In [6]:
def en_uygun_belgeyi_bul(soru):
    soru_vektoru = embed_model.encode(soru, convert_to_tensor=True)

    # Sorunun vektörünü her belgenin vektörüyle karşılaştır
    benzerlikler = util.cos_sim(soru_vektoru, belge_vektorleri)[0]

    # En yüksek benzerliğe sahip belgenin sırasını (index) bul
    en_iyi_index = benzerlikler.argmax().item()

    return belgeler[en_iyi_index], benzerlikler[en_iyi_index].item()

In [8]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_adi = "savasy/bert-base-turkish-squad"

tokenizer = AutoTokenizer.from_pretrained(model_adi)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_adi)

tokenizer_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: savasy/bert-base-turkish-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
def rag_cevapla(soru):
    # 1) Adım: Soruya en uygun belgeyi otomatik bul (retrieval)
    baglam, skor = en_uygun_belgeyi_bul(soru)

    # 2) Adım: Bulunan belgeyi bağlam olarak modele ver, cevabı üret
    sonuc = qa(question=soru, context=baglam)

    print("Soru:", soru)
    print("Seçilen bağlam:", baglam)
    print("Benzerlik skoru:", round(skor, 3))
    print("Cevap:", sonuc["answer"])
    print("-" * 60)

In [14]:
# Verilen bağlamdan cevabı çekip çıkaran fonksiyon
def soru_cevapla(soru, baglam):
    girdiler = tokenizer(soru, baglam, return_tensors="pt")

    with torch.no_grad():
        cikti = qa_model(**girdiler)

    baslangic = torch.argmax(cikti.start_logits)
    bitis = torch.argmax(cikti.end_logits) + 1

    cevap_tokenlari = girdiler["input_ids"][0][baslangic:bitis]
    cevap = tokenizer.decode(cevap_tokenlari, skip_special_tokens=True)
    return cevap


# Retrieval + cevap üretmeyi birleştiren tam RAG fonksiyonu
def rag_cevapla(soru):
    baglam, skor = en_uygun_belgeyi_bul(soru)   # doğru belgeyi otomatik bul
    cevap = soru_cevapla(soru, baglam)          # o belgeden cevabı çek

    print("Soru:", soru)
    print("Seçilen bağlam:", baglam)
    print("Benzerlik skoru:", round(skor, 3))
    print("Cevap:", cevap)
    print("-" * 60)

In [15]:
rag_cevapla("PyTorch'u kim geliştirdi?")
rag_cevapla("Python'u kim geliştirdi?")
rag_cevapla("TensorFlow'u kim geliştirdi?")

Soru: PyTorch'u kim geliştirdi?
Seçilen bağlam: PyTorch, Facebook (Meta) AI Research ekibi tarafından geliştirilen açık kaynaklı bir derin öğrenme kütüphanesidir.
Benzerlik skoru: 0.534
Cevap: Facebook ( Meta ) AI Research ekibi tarafından
------------------------------------------------------------
Soru: Python'u kim geliştirdi?
Seçilen bağlam: Python, 1991 yılında Guido van Rossum tarafından geliştirilen, okunabilir bir programlama dilidir.
Benzerlik skoru: 0.593
Cevap: Guido van Rossum
------------------------------------------------------------
Soru: TensorFlow'u kim geliştirdi?
Seçilen bağlam: TensorFlow, Google Brain ekibi tarafından geliştirilen bir makine öğrenmesi kütüphanesidir.
Benzerlik skoru: 0.651
Cevap: Google Brain ekibi
------------------------------------------------------------


# RAG'i Derinleştirme: Otomatik Belge Getirme (Retrieval) Ekleme

Bu çalışmada, önceki RAG denememizi bir adım ileri taşıdık. Önceki versiyonda bağlamı (context: modele cevap için verdiğimiz metin) **elle** veriyorduk; bu yüzden yanlış bağlam verildiğinde model de yanlış cevap üretiyordu. Bu sürümde ise sisteme, gelen soruya en uygun belgeyi **kendisi bulan** bir getirme adımı (retrieval) ekledik.

## Amaç

- Bir belge havuzu (knowledge base: sorulara cevap ararken bakılan belge topluluğu) oluşturmak.
- Soru geldiğinde, en uygun belgeyi otomatik olarak seçen bir retrieval adımı kurmak.
- Seçilen belgeyi soru-cevap modeline bağlam olarak verip doğru cevabı üretmek.

## Kullanılan Araçlar

- **sentence-transformers** — metinleri embedding'e (embedding: bir metnin anlamını temsil eden sayısal vektör) çeviren kütüphane.
- **Transformers (AutoModelForQuestionAnswering)** — verilen bağlamdan cevabı çeken çıkarımsal (extractive: cevabı uydurmaz, metinden bulup çeker) Türkçe QA modeli.
- **PyTorch** — modellerin arka planda çalıştığı derin öğrenme kütüphanesi.

## Sistemin İşleyişi (İki Aşamalı RAG)

**1. Aşama — Getirme (Retrieval):**
Hem belgeler hem de gelen soru embedding'e çevrilir. Ardından **kosinüs benzerliği** (cosine similarity: iki vektörün anlamca ne kadar benzediğini 0–1 arası ölçen yöntem) ile soru, her belgeyle karşılaştırılır. En yüksek benzerliğe sahip belge, `argmax` (en büyük değerin sırasını bulan işlem) ile seçilir.

**2. Aşama — Cevap Üretme:**
Seçilen belge, QA modeline bağlam olarak verilir. Model, bağlam içinde cevabın başladığı ve bittiği noktaları tahmin ederek (start/end logits) ilgili metin parçasını çeker.

## Sonuç

| Soru | Seçilen Belge | Üretilen Cevap |
|------|---------------|----------------|
| PyTorch'u kim geliştirdi? | PyTorch belgesi | Facebook (Meta) AI Research |
| Python'u kim geliştirdi? | Python belgesi | Guido van Rossum |
| TensorFlow'u kim geliştirdi? | TensorFlow belgesi | Google Brain |

Önceki denemede bağlamı elle verdiğimiz için "PyTorch'u kim geliştirdi?" sorusuna yanlış cevap ("Guido van Rossum") alınıyordu. Bu sürümde sistem, her soru için doğru belgeyi kendisi bulup getirdiğinden bu hata kökten çözüldü.

## Çıkarılan Ders

Gerçek dünyadaki RAG sistemleri de tam bu iki aşamalı mantıkla çalışır: **önce doğru belgeyi bul, sonra o belgeden cevabı üret.** Cevabın kalitesi, büyük ölçüde retrieval adımının doğru belgeyi seçmesine bağlıdır. Belge sayısı büyüdükçe (binlerce/milyonlarca) arama için özel vektör veritabanları kullanılır, ama temel fikir aynıdır.